In [0]:
# ============================================================
# CELL 1: Configuration
# ============================================================

CATALOG       = "telco_customer_churn"
SILVER_TABLE  = f"{CATALOG}.silver.telco_clean"
GOLD_SCHEMA   = "gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

df_silver = spark.table(SILVER_TABLE)
df_silver.createOrReplaceTempView("telco")

print(f"Source: {SILVER_TABLE}")
print(f"Rows  : {df_silver.count():,}")

Source: telco_customer_churn.silver.telco_clean
Rows  : 7,043


In [0]:
# ============================================================
# CELL 2: gold.churn_summary
# ============================================================

churn_summary = spark.sql("""
    SELECT
        Contract,
        tenure_group,
        InternetService,
        COUNT(*)                                    AS total_customers,
        SUM(CAST(Churn_flag AS INT))                AS churned,
        ROUND(AVG(CAST(Churn_flag AS INT)) * 100, 1) AS churn_rate_pct,
        ROUND(AVG(MonthlyCharges), 2)               AS avg_monthly_charges,
        ROUND(SUM(MonthlyCharges), 2)               AS total_mrr
    FROM telco
    GROUP BY Contract, tenure_group, InternetService
    ORDER BY churn_rate_pct DESC
""")

churn_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.churn_summary")

print("✅ gold.churn_summary")
churn_summary.show(10, truncate=False)

✅ gold.churn_summary
+--------------+----------------+---------------+---------------+-------+--------------+-------------------+---------+
|Contract      |tenure_group    |InternetService|total_customers|churned|churn_rate_pct|avg_monthly_charges|total_mrr|
+--------------+----------------+---------------+---------------+-------+--------------+-------------------+---------+
|Month-to-month|New (0-12m)     |Fiber optic    |916            |643    |70.2          |82.08              |75183.15 |
|Month-to-month|Growing (13-24m)|Fiber optic    |425            |215    |50.6          |87.54              |37205.1  |
|Month-to-month|Loyal (49m+)    |No             |2              |1      |50.0          |19.7               |39.4     |
|Month-to-month|Mature (25-48m) |Fiber optic    |521            |226    |43.4          |91.24              |47535.45 |
|Month-to-month|New (0-12m)     |DSL            |690            |293    |42.5          |47.88              |33039.7  |
|Month-to-month|Loyal (49m+

In [0]:
# ============================================================
# CELL 3: gold.revenue_at_risk
# ============================================================

revenue_at_risk = spark.sql("""
    SELECT
        Contract,
        monthly_segment,
        COUNT(*)                                         AS total_customers,
        SUM(CASE WHEN Churn_flag THEN 1 ELSE 0 END)     AS churned_customers,
        ROUND(SUM(MonthlyCharges), 2)                    AS total_mrr,
        ROUND(SUM(CASE WHEN Churn_flag 
                  THEN MonthlyCharges ELSE 0 END), 2)    AS mrr_at_risk,
        ROUND(SUM(CASE WHEN Churn_flag 
                  THEN MonthlyCharges ELSE 0 END) 
              / SUM(MonthlyCharges) * 100, 1)            AS pct_revenue_at_risk
    FROM telco
    GROUP BY Contract, monthly_segment
    ORDER BY mrr_at_risk DESC
""")

revenue_at_risk.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.revenue_at_risk")

print("✅ gold.revenue_at_risk")
revenue_at_risk.show(10, truncate=False)

✅ gold.revenue_at_risk
+--------------+---------------+---------------+-----------------+---------+-----------+-------------------+
|Contract      |monthly_segment|total_customers|churned_customers|total_mrr|mrr_at_risk|pct_revenue_at_risk|
+--------------+---------------+---------------+-----------------+---------+-----------+-------------------+
|Month-to-month|Premium (>$85) |1166           |612              |111443.95|58391.15   |52.4               |
|Month-to-month|High ($65-85)  |1129           |575              |85652.05 |43850.45   |51.2               |
|Month-to-month|Mid ($35-65)   |876            |297              |44359.65 |14578.55   |32.9               |
|One year      |Premium (>$85) |501            |102              |50280.8  |10423.7    |20.7               |
|Month-to-month|Low (<$35)     |704            |171              |15838.5  |4026.95    |25.4               |
|Two year      |Premium (>$85) |529            |31               |53940.15 |3258.1     |6.0              

In [0]:
# ============================================================
# CELL 4: gold.customer_360
# ============================================================

customer_360 = spark.sql("""
    SELECT
        customerID,
        gender,
        SeniorCitizen,
        Partner,
        Dependents,
        tenure,
        tenure_group,
        Contract,
        InternetService,
        MonthlyCharges,
        TotalCharges,
        monthly_segment,
        Churn,
        Churn_flag,

        -- Lifetime Value 
        ROUND(MonthlyCharges * tenure, 2)           AS estimated_ltv,

        -- Risk Score 
        CASE
            WHEN Churn_flag = true                  THEN 'Churned'
            WHEN Contract = 'Month-to-month' 
             AND tenure <= 12                       THEN 'High Risk'
            WHEN Contract = 'Month-to-month'        THEN 'Medium Risk'
            ELSE                                         'Low Risk'
        END                                         AS risk_category

    FROM telco
""")

customer_360.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.customer_360")

print("✅ gold.customer_360")
customer_360.show(5, truncate=False)

✅ gold.customer_360
+----------+------+-------------+-------+----------+------+---------------+--------------+---------------+--------------+------------+---------------+-----+----------+-------------+-------------+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|tenure_group   |Contract      |InternetService|MonthlyCharges|TotalCharges|monthly_segment|Churn|Churn_flag|estimated_ltv|risk_category|
+----------+------+-------------+-------+----------+------+---------------+--------------+---------------+--------------+------------+---------------+-----+----------+-------------+-------------+
|7590-VHVEG|Female|No           |Yes    |No        |1     |New (0-12m)    |Month-to-month|DSL            |29.85         |29.85       |Low (<$35)     |No   |false     |29.85        |High Risk    |
|5575-GNVDE|Male  |No           |No     |No        |34    |Mature (25-48m)|One year      |DSL            |56.95         |1889.5      |Mid ($35-65)   |No   |false     |1936.3       |Low Risk     |


In [0]:
# ============================================================
# CELL 5: Final check — Gold tables
# ============================================================

gold_tables = ["churn_summary", "revenue_at_risk", "customer_360"]

print("=" * 45)
print("GOLD LAYER — FINAL VERIFICATION")
print("=" * 45)

for table in gold_tables:
    count = spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{GOLD_SCHEMA}.{table}") \
                 .collect()[0][0]
    print(f"✅ gold.{table}: {count:,} rows")

print("\n── Top 5 Highest Churn Rates ──")
spark.sql(f"""
    SELECT Contract, tenure_group, churn_rate_pct, total_customers
    FROM {CATALOG}.{GOLD_SCHEMA}.churn_summary
    ORDER BY churn_rate_pct DESC
    LIMIT 5
""").show(truncate=False)

GOLD LAYER — FINAL VERIFICATION
✅ gold.churn_summary: 35 rows
✅ gold.revenue_at_risk: 12 rows
✅ gold.customer_360: 7,043 rows

── Top 5 Highest Churn Rates ──
+--------------+----------------+--------------+---------------+
|Contract      |tenure_group    |churn_rate_pct|total_customers|
+--------------+----------------+--------------+---------------+
|Month-to-month|New (0-12m)     |70.2          |916            |
|Month-to-month|Growing (13-24m)|50.6          |425            |
|Month-to-month|Loyal (49m+)    |50.0          |2              |
|Month-to-month|Mature (25-48m) |43.4          |521            |
|Month-to-month|New (0-12m)     |42.5          |690            |
+--------------+----------------+--------------+---------------+

